In [ ]:
import sys 
sys.path.append("../")
from utils.null_analyzer import NullAnalyzer
import pandas as pd
import numpy as np
from utils.target_var_inspection import plot_kde, plot_barchart
from utils.target_var_inspection import plot_log_transformed_distr
from utils.neighborhood_cleaning import normalize_neigh_col
import statsmodels.api as sm


# Demand Decomposition of Short Term Rentals in Lombardy

## Introduction

For decades, most of the accommodation available for tourists visiting a city was provided by specialized structures like hotels. In the past few years, companies like Airbnb completely changed this paradigm, allowing regular people to rent out their apartments to tourists. <br> <br>
While this opportunity allowed many hosts to profit from their previously empty houses, it has also reshaped the neighborhood dynamics and composition of many cities, such as Milan, Bergamo and others, causing traditional businesses to disappear, while tourist oriented businesses proliferated. <br> <br>
While the topic itself can be seen as controversial, we find the underlying dynamics that power it very interesting, and we aim to explore them with this project.
<br><br>

The aim of the project is to identify the key drivers of demand for short term rentals in Lombardy, specifically in the cities of Milan and Bergamo. <br>
We set out to determine the specific contribution that factors like neighborhood (Duomo, Brera,...), as well as city itself (Milan vs Bergamo), as well as price, room type (Apartment, Private Room, ...) have on the final demand for the listing. Crucially, we aim to also estimate the price elasticity of demand in order to get some insight into tourist behavior.<br>
<br>
We use a dataset downloaded from **Inside Airbnb**, which contains a scraping of all the listings available on Airbnb as of the 27th September 2025. <br>

## Data Loading

Before starting the data exploration, we report some of the known limitations of the dataset, also highlighted on the **Inside Airbnb** website. <br>
- When building the dataset, it was not possible to differentiate between a "booked" listing and an unavailable (e.g. blocked by the host) listing. The authors therefore simply reported the availability rate, intended as the number of nights in which the room can be booked.
- There is no historic data about past occupancy and past revenue generated, therefore the authors use some proxy to provide an estimate based on the number of reviews received by the listing.

In [ ]:
df_mi = pd.read_csv("../../data/listings_milan.csv")
df_bg = pd.read_csv("../../data/listings_bergamo.csv")

df_mi.head()

In [ ]:
df_mi.info()

In [ ]:
df_bg.info()

We notice that both dfs have the same colums, and that the Milan listings are 6x more than the Bergamo ones. <br>
Additionally, while many columns have no null values, some crucial ones like **price** have null values that will need to be addressed.

We start by joining the two dfs together, adding a column that identifies in which city is each listing.

In [ ]:
## Adding an unambiguous first column to avoid collisions later
df_mi["unique_id"] = df_mi["id"].astype(str) + "_mi"
df_mi["city"] = "MI"
df_bg["unique_id"] = df_bg["id"].astype(str) + "_bg"
df_bg["city"] = "BG"


n_rows_mi = df_mi.shape[0]
n_rows_bg = df_bg.shape[0]

column_order = ["unique_id", "city"] + [col for col in df_mi.columns if col not in ("unique_id", "city")]
df_mi = df_mi[column_order]
df_bg = df_bg[column_order]

# Merging the dfs 
df = pd.concat([df_mi, df_bg], axis=0, ignore_index=True)
assert n_rows_bg + n_rows_mi == df.shape[0] # Verifying all rows are preserved

## Column Selection

As shown above, the original dfs have 78 columns each, most of which have little use for our specific project. Therefore, we select the most useful ones for our use case, and we detail their natura and use in this cell.
- **unique_id**: key of the dataframe
- **city**: column added during the df union, indicates in which city is each listing
- **host_since**: date indicating when the host joined the platform. Useful as proxy for whether a listing is new or not.
- **host_response_rate**: percentage indicating the fraction of inquiries that were addressed by the host
- **host_response_time**: categorical column indicating how fast a host answers inquiries
- **minimum_nights**: minimum number of nights that must be booked in the listing
- **number_of_reviews_l30d**: number of reviews received by the listing in the last 30 days
- **number_of_reviews_ly**: number of reviews received by the listing during the past year
- **first_year**: date of the first review received by the listing
- **last_review**: date of the last review received by the listing
- **availability_30**: number of days currently not available out of the next 30
- **availability_60**: number of days currently not available out of the next 60
- **availability_90**: number of days currently not available out of the next 90. This column will serve as the proxy for the demand.
- **availability_365**: number of days currently not available out of the next 365
- **accommodates**: number of guests that can be hosted in the listing
- **is_host_superhost**: whether the host has received the superhost badge. The badge is obtained after having received a high customer volume, high ratings, low cancellation rates and having had high response rates. I will use it as proxy for host quality.
- **neighborhood_cleansed**: in Milan, it indicates the city neighborhood where the listing is. In Bergamo, it indicates the town where the listing is (i.e. Bergamo, Seriate,...)
- **room_type**: type of listing, ranging from full apartment to private room
- **price**: price in dollars of the room per night as of the 27th of September


In [ ]:
necessary_cols = [
    "unique_id",
    "city",
    "host_since",
    "host_response_rate",
    "host_response_time",
    "minimum_nights",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "first_review",
    "last_review",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "accommodates",
    "host_is_superhost",
    "neighbourhood_cleansed",
    "room_type",
    "price"]

df = df[necessary_cols]

We verify the integrity of all the relevant columns of the df.

In [ ]:
## Running check on NULL entries
null_analyzer = NullAnalyzer(threshold=5, stratify_col="city")

null_analyzer.check_nulls_globally(df=df)
null_analyzer.check_nulls_strat(df=df, delta_threshold=5)

We fill the nulls in all columns except the price one. We also add flags to record which hosts never had reviews or were never contacted.

In [ ]:
fill_values = {
    "host_since": "2000-01-01",               
    "host_response_rate": "0%",               
    "host_response_time": "unknown",
    "minimum_nights": 1,
    "maximum_nights": 365,
    "number_of_reviews_l30d": 0,
    "number_of_reviews_ly": 0,
    "first_review": "2000-01-01",             
    "last_review": "2000-01-01",              
    "availability_30": 0,
    "availability_60": 0,
    "availability_90": 0,
    "availability_365": 0,
    "accommodates": 1,
    "host_is_superhost": "f",
    "neighbourhood_cleansed": "unknown",
    "room_type": "unknown",    
} 


df["has_host_been_contacted"] = df["host_response_rate"].isna()
df["host_never_had_reviews"] = df["first_review"].isna()
df["host_without_join_date"] = df["host_since"].isna()

n_before = df.shape[0]
df = df.fillna(value=fill_values).dropna()
n_after = df.shape[0]
print(f"Rows before: {n_before}, rows after: {n_after}, dropped rows: {(n_before - n_after)/n_before:.2%}")

### Scope Definition

In this subsection, we filter out listings that do not fall within the scope of this project. For instance, we will investigate the following characteristics:
- how old/new a listing is
- whether the listing is long or short term

In [ ]:
snapshot_date = pd.to_datetime(df["last_scraped"].iloc[0] if "last_scraped" in df.columns else "2025-09-27")
df["host_age_days"] = (snapshot_date - pd.to_datetime(df["host_since"], errors="coerce")).dt.days

In [ ]:
plot_kde(df[~df["host_without_join_date"]], target_column="host_age_days", city_column="city", rng=(0, 10000))

We clearly see that most hosts joined the platform more than 1y ago. Since we do not have the listing initial date available, we use the host join date as proxy for listing insertiond date, and remove all listings whose host joined the platform less than one year ago. <br><br>
We perform this action to ensure that listings have been operational for a sufficient time and do not benefit from initial-only effects. <br><br>
We remark that the choice of the proxy is very conservative, and leaves in the df all the listings that are new, but belong to a host that had previous listings.

In [ ]:
n_before = df.shape[0]
df = df[df["host_age_days"] >= 365]
n_after = df.shape[0]
print(f"Filtered out {(n_before - n_after)/n_before:.2%} listings with host age < 1 year")

# Building occupancy features
occupancy_windows = [30, 60, 90, 365]
for window in occupancy_windows:
    df[f"occupancy_{window}"] = 1 - (df[f"availability_{window}"] / window)

In [ ]:
plot_kde(df, target_column="minimum_nights", city_column="city", rng=(0, 50))

The minimum nights variable exhibits a clear bimodal distribution, with a peak around 2/3 days likely indicating short term renters, and another one around 30 days likely indicating long term renters. <br><br>
For the purpose of this project, we can remove the long term renters. 

In [ ]:
# Restricting the scope to short term rentals
n_before = df.shape[0]
df = df[df["minimum_nights"] < 15]
n_after = df.shape[0]

print(f"Filtered out {(n_before - n_after)/n_before:.2%} listings with minimum_nights >= 15")
print(f"Retained {n_after} listings with minimum_nights < 15")

## Analysis of Occupancy Variable (Dependent Variable)

The main objective of the analysis is to estimate the demand of listings through multivariate regression. We use the occupancy rate in the next 90 days as proxy for the demand. <br><br>
The reason for this choice is that we find that it is a sufficiently long period of time to average out noise, while it is not long enough to cause most listings to have a very low occupancy rate. <br>
We have that  $Occupancy \in [0, 1]$, but we need a well behaved distribution to perform an accurate regression. In this subsection, we inspect the distribution and perform the necessary cleaning.

In [ ]:
## VERIFYING WHETHER MILAN AND BERGAMO TARGET VARIABLES ARE DISTRIBUTED DIFFERENTLY
plot_kde(df, target_column="occupancy_90", city_column="city", rng=(0, 1))

We notice that both Milan and Bergamo have awkward distributions, with peaks around $Occupancy = 1$, signalling an unusually high number of fully booked listings <br>
We also identify slight peaks around $Occupancy = 0.33$ and $Occupancy = 0.66$, potentially signalling peaks in listings that are fully booked in the 30 or 60 days. <br><br>
As described at the beginning, our occupancy metric is computed from the author's availability metric, which only computes the bookable nights, without making a distinction as to why a night is not bookable.

We hypothesize that there are two underlying phenomena that are simultaneusly distorting the data:
- Listings Blocked by Hosts
- Listings not managed by hosts
<br>
Blocked listings might happen when hosts set them as unavailable for the upcoming 90 days. The scraping identifies these bookings are occupied because the room is indeed __not available__, but clearly these listings are not meaningful for the analysis. <br><br>
On the other hand, listings that are not managed are listings that never receive any traffic and that likely did not attract any client in a long time, potentially also because of bad management on the host's side. 

### Data Refinement

In order to remove all the listings which are 100% occupied because they are blocked by the hosts, we must find distinctive characteristics that separate true best sellers from blocked entries. <br> <br>
We start by inspecting the host response rate and time, under the assumption that true top sellers will have hosts that have top performance in these two fields.

In [ ]:
plot_barchart(df, category_col="host_response_time")

In [ ]:
df["host_response_rate"] = df["host_response_rate"].astype(str).str.rstrip("%")

df["host_response_rate"] = pd.to_numeric(
    df["host_response_rate"], errors="coerce"
).astype("Int64")

plot_kde(df[~df["has_host_been_contacted"]], target_column="host_response_rate", rng=(0, 100))

The overwhelming majority of hosts in both cities responds "within an hour" and has response rate above 90%. It is safe to assume that true top sellers will have hosts that respond within an hour and with response rate above 90%.

In [ ]:
df["is_host_inactive"] = (df["host_response_rate"] < 90) | (df["host_response_time"] != "within an hour")

In [ ]:
# Constructing features to capture host inactivity and blocked listings
 
df["last_review_dt"] = pd.to_datetime(df["last_review"], errors="coerce")
df["first_review_dt"] = pd.to_datetime(df["first_review"], errors="coerce")

df["reviews_per_year"] = pd.to_numeric(df["number_of_reviews_ly"], errors="coerce")
df["availability_365"] = pd.to_numeric(df["availability_365"], errors="coerce")

df["is_host_inactive"] = df["is_host_inactive"].astype(bool)
df["host_never_had_reviews"] = df["host_never_had_reviews"].astype(bool)

df["occupancy_90"] = pd.to_numeric(df["occupancy_90"], errors="coerce")
df["occupancy_60"] = pd.to_numeric(df["occupancy_60"], errors="coerce")
df["occupancy_30"] = pd.to_numeric(df["occupancy_30"], errors="coerce")


We use the following thresholds to define "high occupancy":
- For the 90 days, a 90% occupancy is considered high
- For the next 60 days, a 90% occupancy is considered high
- For the next 30 days, a 95% occupancy is considered high

Additionally, we consider as very unlikely that a true top-selling listing had few reviews in the past year. <br>
We use the following formula to compute a very conservative estimate of the number of reviews a top-selling listing must have received, also accounting for seasonality and potential closures. 
$$ \frac{time\_window * occupancy\_rate}{avg\_stay\_duration} * review\_rate $$
Where the fraction computes the expected number of guests the listing had in the 90 days time window last year, and the review rate maps that number into the expected number of reviews. <br><br>
This formula is very conservative in the sense that assumes that the listing last year was live for at least the time window for which is highly booked now, and estimates the number of reviews it would have obtained last year. <br><br>
Using a the very conservative $review\_rate = 50\%$ and $avg\_stay\_duration = 4 nights$, we obtain the following results:
- 10 reviews for 90 day period
- 7 reviews for 60 day period
- 4 reviews for 30 day period

In [ ]:
high_occupancy_90 = df["occupancy_90"] > 0.90
high_occupancy_60 = df["occupancy_60"] > 0.90
high_occupancy_30 = df["occupancy_30"] > 0.95


block_conditions = [
    high_occupancy_90,
    high_occupancy_60,
    high_occupancy_30
]

review_floor_choices = [10, 7, 4]

df["required_review_floor"] = np.select(block_conditions, review_floor_choices, default=0)

We consider a listing blocked if the listing is high demand and one of the following conditions holds:
- **The host has not had a single review in the last 30 days**. We find it unlikely that a listing that had no review for a month is not fully booked
- **The host had less reviews than the threshold set before**. We find it unlikely that a fully booked listing had very little reviews in the past year.
- **The host has 0% availability for the whole year**. We find it unlikely that any listing is fully booked one year in advance.
- **The host is either inactive or never had any review**. We find it unlikely that a low quality host can have a fully booked listing.


In [ ]:
cond_review_stale = (snapshot_date - df["last_review_dt"]).dt.days > 30       # hasn't been reviewed in a month

cond_low_momentum = (df["reviews_per_year"] < df["required_review_floor"]) 
cond_calendar_dead = df["availability_365"] == 0
cond_host_dead = (df["is_host_inactive"] | df["host_never_had_reviews"]) == True

any_trigger_tripped = (
    cond_review_stale | 
    cond_low_momentum | 
    cond_calendar_dead | 
    cond_host_dead
)

df["block_90"] = high_occupancy_90 & any_trigger_tripped
df["block_60"] = high_occupancy_60 & any_trigger_tripped & ~high_occupancy_90
df["block_30"] = high_occupancy_30 & any_trigger_tripped & ~high_occupancy_60 & ~high_occupancy_90

df["is_blocked"] = df["block_90"] | df["block_60"] | df["block_30"]

In [ ]:
n_before = df.shape[0]
df = df[~df["is_blocked"]]
n_after = df.shape[0]

print(f"Filtered out {(n_before - n_after)/n_before:.2%} listings flagged as blocked")
print(f"Retained {n_after} listings not flagged as blocked")

In [ ]:
plot_kde(df, target_column="occupancy_90", city_column="city", rng=(0, 1))

We find that the distributions are now much smoother, and there is no bump around 100% occupancy, signalling that indeed many listings were blocked by hosts.

We now tackle the issue of dead listings, which are likely not managed. <br>
We choose to remove listings that have 0 nights booked in the next 90 days and that had 0 reviews in the past 180 days.

In [ ]:
cond_listing_dead = ((snapshot_date - df["last_review_dt"]).dt.days > 180) & (df["occupancy_90"] == 0)
cond_listing_not_managed = (df["is_host_inactive"] & (df["occupancy_90"] == 0))
n_before = df.shape[0]
df = df[~(cond_listing_dead | cond_listing_not_managed)]
n_after = df.shape[0]
print(f"Filtered out {(n_before - n_after)/n_before:.2%} listings flagged as potentially dead")
print(f"Retained {n_after} listings not flagged as potentially dead")

In [ ]:
plot_kde(df, target_column="occupancy_90", city_column="city", rng=(0, 1))

## Log-Transformation of Target Variable

As clarified in the introduction, we are interested in estimating the price elasticity of demand. In order to do so, we will select as independent variable $\log(Price)$, and as dependent variable $\log(Occupancy)$, so that the regression coefficient will represent elasticity. <br>
For this reason, we also inspect the distribution of the log transformed occupancy.

In [ ]:
plot_log_transformed_distr(df, variable_col="occupancy_90", city_col="city")

While Milan exhibits a nice distribution, Bergamo is dominated by values that are very close to 0, signalling a fundamental difference between the two markets. <br>
While the Milan market is very mature and highly booked year-round, Bergamo shows a very high number of not booked listings which were not eliminated by the filters, and which are likely seasonal listings that are popular during summer and are abandoned during winter, when the turist influx stops.

## Price Analysis

As mentioned above, we will use $\log(Price)$ in the regression, therefore we inspect its distribution

In [ ]:
df["price"] = df["price"].str.replace("$", "", regex=False)
df["price"] = df["price"].str.replace(",", "", regex=False)
df["price"] = df["price"].astype(float).astype(int)

In [ ]:
plot_kde(df, target_column="price", city_column="city", rng=(0, 1000))

As well as its log-transformed distribution

In [ ]:
plot_log_transformed_distr(df, variable_col="price", city_col="city")

The log-transformed price distributions are very close to a normal.

## Neighborhood Analysis

We plan to use the different neighborhoods as categorical variables in the regression, to inspect their respective impacts on the baseline demand. <br>
In this subsection, we inspect and clean the variable.

In [ ]:
print("Most Popular Neighbourhoods (MI):")
print(df["neighbourhood_cleansed"][df["city"] == "MI"].value_counts().head(10))

print("\nLeast Popular Neighbourhoods (MI):")
print(df["neighbourhood_cleansed"][df["city"] == "MI"].value_counts().tail(10))

print("Most Popular Neighbourhoods (BG):")
print(df["neighbourhood_cleansed"][df["city"] == "BG"].value_counts().head(10))

print("\nLeast Popular Neighbourhoods (BG):")
print(df["neighbourhood_cleansed"][df["city"] == "BG"].value_counts().tail(10))

We see that most of the neighborhoods are very poorly populated, therefore we choose to group them in the following way:
- For Bergamo, if the neighborhood is "Bergamo" (i.e. the City), then call it "Bergamo_City", otherwise call it "Bergamo_Province"
- For Milan, retain only neighborhoods with more than 100 listins, and aggregate the others into "Other_Milan"

In [ ]:
df = normalize_neigh_col(df, threshold=100)

In [ ]:
plot_barchart(df, category_col="neighbourhood")

## Room Type Analysis

We also want to use the Room Type as categorical variable in the regression, therefore we inspect it.

In [ ]:
plot_barchart(df, category_col="room_type")

Since "Shared room" and "Hotel room" have less than 1% of listings, we simply drop these rows and will verify how much renting a room modifies demand with respect to an entire home/apartment.

In [ ]:
df = df[df["room_type"].isin(["Entire home/apt", "Private room"])]

## Accommodates Analysis

Another interesting variable we will use in the regression is the number of guests a listing can accommodate.

In [ ]:
plot_barchart(df, category_col="accommodates")

In [ ]:
print(df["accommodates"].value_counts(normalize=True).sort_index().cumsum())

Listings which can accommodate 8 people or less cover more than 98% of the total listings. Since we are interested in the core short term rental market, which is mostly made of one-family rooms/houses, we can safely drop the listings which host more than 8 people.

In [ ]:
df = df[df["accommodates"] <= 8]

# Superhost Analysis

Since the dataset had no feature that directly mapped to "being a good host", we use the superhost indication of a host as proxy for their professionality.

In [ ]:
plot_barchart(df, category_col="host_is_superhost")

The two categories are already well represented, so no further action is needed

# Model Fitting

## Data Transformation

In [ ]:
df = df.reset_index(drop=True)

# Creating Log transformed variables
df["occupancy_adj"] = np.clip(df["occupancy_90"], 0.001, 1.0) # to ensure no 0 values for log transformation
df["log_occupancy"] = np.log(df["occupancy_adj"])
df["price_adj"] = np.clip(df["price"], 1, None) 
df["log_price"] = np.log(df["price_adj"])

# Creating binary variable for superhost status
df["is_superhost"] = (df["host_is_superhost"] == "t").astype(int)

room_type_dummies = pd.get_dummies(df["room_type"], prefix="room", drop_first=True, dtype=int)
neighbourhood_dummies = pd.get_dummies(df["neighbourhood"], prefix="neigh", drop_first=True, dtype=int)

X = df[["log_price", "accommodates", "is_superhost"]].join([room_type_dummies, neighbourhood_dummies])

room_categories = sorted(df["room_type"].unique())
geo_categories = sorted(df["neighbourhood"].unique())

print(f"Room Type that consistutes the baseline: {room_categories[0]}")
print(f"Neighborhood that constitutes the baseline: {geo_categories[0]}")

## Model Definition

In [ ]:
# Adding intercept term for regression
X = sm.add_constant(X)
Y = df["log_occupancy"]

In [ ]:
# We use robust standard errors (HC3) to account for potential heteroskedasticity in the residuals (especially for Bergamo)
model = sm.OLS(Y, X).fit(cov_type="HC3")
print(model.summary())

# Results

The results clearly show that the regression is mathematically sound and successful. <br>
- The F-statistic has value of 31.04, and the p-value is essentially 0, which in turn mean that there is strong statistical significance that the independent variables have a relationship with the occupancy in the upcoming 90 days.
- The $R^2$ and $R^2_adj$ scores, respectively 0.126 and 0.123, confirm that the model is able to explain roughly 12.6% of the total variance, and the independent variables are all meaningful (otherwise the $R^2_adj$ would have dropped by more than 0.003)
- **price, the number of possible guests, whether the host was a superhost and the room type all have near 0 p-values**. This means that they are all highly statistically significant.
- **The price coefficient represents elasticity**: since the coefficient is -0.35, we can deduce that a 1% increase in price will reduce in a -0.35% reduce in occupancy in the next 90 days. Therefore we can classify the demand as inelastic, and tourists will likely absorb price increases and not change their booking behavior. 
- **accommodates has a log-linear relationship with the occupancy**: the coefficient is 0.0687, meaning that if the house can accommodate 1 more guest, the demand increases by 6.87% (holding all other variables constant).
- **being a high-quality host (superhost) is associated with a strong demand**: the coefficient of 0.2156 implies that being a superhost is associated with a demand higher by $100 \times (e^\beta-1) = 24\%$. 
- **renting a private room instead of an entire apartment decreases demand**: the coefficient of −0.1793 implies that private rooms have 15% less demand than entire apartments (holding all other variables constant).
<br>

Regarding neighborhoods, we make some interesting findings:
- The baseline neighborhood that was omitted from the regression is "Affori", so all the coefficients are computed relative to it.
- Once price, room type, number of guests and host quality is held constant, the demand for listings in the city of Bergamo is not statistically distinguishable from that of Affori in Milan. 
- Crucially, moving the listing outside the city of Bergamo and into the provice towns yields a statistically significant coefficent of -1.03, which implies a 64% decrease in demand. This clearly shows the demand difference between listings in Bergamo and outside of it, as well as differences between non-central parts of Milan and non-central parts of Bergamo. 
- **Duomo, Brera and Centrale all have statistically significant, strongly positive coefficients, indicating that listings in these neighborhoods experience strong demand**. For instance, we see the following increase: Duomo +73%, Brera + 48%, Centrale +31%.
- **Dergano, Villapizzone and Lodi - Corvetto experience statistically significant negative coefficients, which indicate weaker demand than the baseline**. In particular, we observe Dergano -31%, Villapizzone -23% and Lodi - Corvetto -20%.
<br><br>
Overall, we find the results credible, as they highlight that notoriously premium neighborhoods of Milan such as Brera and Duomo experience higher demand overall, while less premium neighborhoods like Corvetto and Lodi experience lower demand. We find the price elasticity results extremely interesting because they show that tourists are inelastic to price changes.

# Limitations

In this section, we identify the limitations of our work:
- We did not have the true occupancy of each listing available, therefore we had to refine the variable through heuristics and proxies. The filtered dataset can still contain out of scope listings that pollute the result.
- We did not have the activity status of a listing, so we could not know whether a listing was empty because of actually low demand, or because the host does not manage the listing. 
- We had a very limited set of features available, so the model explains only a fraction of the total variance.

# Next Steps

While the regression worked well and the results are extremely interesting, we notice that $R^2 = 0.126$, meaning that the model only explains 12.6% of the total variance of the data. <br><br>
While this is expected given the complexity of the data, I propose three key directions that I believe are worth pursuing, ordered from the most to the least time effective:
- **Use amenities as additional categorical features**. Indeed, being close to the metro station or having a kitchen are key features in a listing, and can dramatically shift the demand for it. 
Selecting a few key amenities and using them as variables in the regression could increase the variable explained by the model.
- **Use listing name and description as feature**: The listing names and description are the first things a user sees on the website, therefore they play a crucial role in deciding the demand of a listing. 
A feasible improvement could consist of processing these names and descriptions with an LLM, asking the LLM to rank them on a scale from 1 to 5 depending on some criteria such as completeness, elegance and other KPIs. The socres can then be used to improve the regression.
- **use the listing image as feature**: another key aspect of a listing is the cover image. A possible improvement could be letting a Vision Model process the images, assigning a score to each one or computing a vectorized representation of the objects present in the image. The scores or the vector of objects could once again be used in the regression.